# Radio Bulletin Generator — Three Languages

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Generate natural, radio-ready weather bulletins in **three languages**:
1. **English** (Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

- **Target length**: ~300 words (~2 minutes at 150 wpm radio reading pace)
- **Style**: Philippine radio broadcast — authoritative, clear, flowing prose
- **Model**: Gemma 4 E4B via Ollama (text-only — no vision needed here)
- **Input**: Full markdown bulletin text (from notebook 04)
- **Output**: Plain text scripts saved to `data/radio_bulletins/`

### Two sample bulletins
1. `PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito
2. `PAGASA_22-TC02_Basyang_TCA#01` — Tropical Cyclone Alert, Basyang

**Total output**: 6 scripts (2 bulletins × 3 languages)

In [1]:
import json
import requests
import time
from pathlib import Path

OLLAMA_API = "http://localhost:11434"
MODEL_NAME = "gemma4:e4b"
TIMEOUT = 10 * 60  # 10 minutes

data_dir = Path("../data")
markdown_dir = data_dir / "gemma4_results"
output_dir = data_dir / "radio_bulletins"
output_dir.mkdir(exist_ok=True)

# The two bulletins we are working with
SAMPLES = [
    "PAGASA_20-19W_Pepito_SWB#01",
    "PAGASA_22-TC02_Basyang_TCA#01",
]

# Verify Ollama is running
try:
    r = requests.get(f"{OLLAMA_API}/api/tags", timeout=5)
    names = [m['name'] for m in r.json()['models']]
    status = "\u2713" if any(MODEL_NAME in n for n in names) else "\u26a0\ufe0f  model not found"
    print(f"\u2713 Ollama running \u2014 {MODEL_NAME}: {status}")
except Exception as e:
    print(f"\u2717 Ollama not reachable: {e}")

print(f"\u2713 Output dir: {output_dir.absolute()}")


✓ Ollama running — gemma4:e4b: ✓
✓ Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins


## 1. Load Bulletin Markdown

Load the raw markdown text extracted by notebook 04. This is the primary input — no structured JSON used.

In [2]:
def load_bulletin(stem: str) -> dict:
    """Load the raw markdown for a bulletin stem (primary input for generation)."""
    markdown_path = markdown_dir / f"{stem}_markdown.md"

    if not markdown_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {markdown_path}")

    markdown = markdown_path.read_text(encoding="utf-8")

    return {"stem": stem, "markdown": markdown}


bulletins = [load_bulletin(s) for s in SAMPLES]

for b in bulletins:
    print(f"\n{b['stem']}")
    print(f"  Markdown: {len(b['markdown'])} chars")
    print(f"  Preview:  {b['markdown'][:120].strip()!r}")



PAGASA_20-19W_Pepito_SWB#01
  Markdown: 4825 chars
  Preview:  '## REPUBLIC OF THE PHILIPPINES\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PHILIPPINE ASTROPHYSICAL, GEOPHYSICAL AND ASTR'

PAGASA_22-TC02_Basyang_TCA#01
  Markdown: 3224 chars
  Preview:  '# PHILIPPINE ARCHIPELAGO\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PAGASA**\nPhilippine Atmospheric, Geophysical and Ast'


## 2. Radio Bulletin Prompts — Three Languages

Generate radio bulletins in:
1. **English** (standard Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

All versions maintain the same style:
- Flowing prose — no bullet points, no tables, no markdown

- ~300 words (~2 minutes reading time)- Clear structure: opening → current situation → forecast → affected areas → public safety → closing

- Repeat storm name and signal areas (radio listeners may tune in mid-broadcast)- Plain spoken numbers

In [3]:
PROMPTS = {
    "en": {
        "system": """You are converting a PAGASA typhoon bulletin into a plain spoken weather announcement in English.

STYLE:
- Write as if explaining to a neighbour — conversational, simple, direct
- No broadcaster language, no formal sign-offs, no station IDs, no "Good morning listeners"
- Short sentences. Common words. Anyone should understand this.
- Use digits for numbers (e.g. "25 kilometres per hour", "Signal 2")
- Cover: what the storm is, where it is, where it is going, who is affected, what people should do
- DO NOT add information that is not in the original bulletin

PHILIPPINE PLACE NAME PRONUNCIATION — spell these phonetically so they are read correctly by the TTS engine:
  - Catanduanes → ka-tan-du-a-nes
  - Cagayan → ka-ga-yan
  - Isabela → i-sa-be-la
  - Visayas → bi-sa-yas
  - Mindanao → min-da-naw
  - Quezon → ke-son
  - Batanes → ba-ta-nes
  - Samar → sa-mar
  - Leyte → ley-te
  - Palawan → pa-la-wan
  - Bicol → bi-kol
  - Masbate → mas-ba-te
  - Surigao → su-ri-ga-o
  - Pangasinan → pa-nga-si-nan
  - Zamboanga → zam-bo-an-ga

FORMATTING: Plain flowing prose only. No headings, no bullet points, no bold, no markdown. Paragraph breaks (blank lines) between ideas.

LENGTH: About 300 words.""",

        "user_template": (
            "Convert this PAGASA weather bulletin into a plain conversational English announcement.\n\n"
            "{markdown}\n\n"
            "Write the announcement now. Conversational tone, simple words, about 300 words. "
            "No headings, no bullet points, no markdown. Spell Filipino place names phonetically."
        ),
    },

    "tl": {
        "system": """Ikaw ay nagsusulat ng simpleng pahayag tungkol sa isang malakas na bagyo para marinig ng mga tao.

ESTILO — PURO TAGALOG, WALANG INGLES:
- Magsulat na parang nagkukwento ka sa isang kapitbahay — simple, natural, walang paligoy-ligoy
- BAWAL ang mga salitang Ingles maliban sa mga pangalan ng tao at lugar (hal. Pepito, Catanduanes, Luzon)
- Gamitin ang pang-araw-araw na Tagalog — hindi pormal, hindi opisyal, hindi balita sa TV
- Kung may teknikal na salita, i-spell phonetically gamit ang gitling:
    tai-pun, tro-pi-kal di-pre-syon, sig-nal nam-ber wan, ki-lo-me-tro ba-wat o-ras,
    storm serj, land-pol, kos-tal, pag-asa, pore-kast, wor-ning, ad-bay-so-ri
- Maikling pangungusap. Madaling salita. Dapat maintindihan ng sinumang Pilipino.
- Gamitin ang mga digit para sa mga numero (hal. 25 kilometro bawat oras, Signal 2)
- Sabihin: ano ang bagyo, nasaan ito, saan patungo, sino ang maaapektuhan, ano ang dapat gawin
- HUWAG magdagdag ng impormasyon na wala sa orihinal na bulletin

FORMATTING: Natural na daloy ng prosa. Walang headings, walang bullets, walang bold, walang markdown. Blank lines sa pagitan ng mga talata.

HABA: Mga 300 salita.""",

        "user_template": (
            "I-convert ang PAGASA weather bulletin na ito sa simpleng pahayag sa Tagalog.\n\n"
            "{markdown}\n\n"
            "Isulat ang pahayag ngayon. Puro Tagalog — walang Ingles maliban sa mga pangalan. "
            "Natural na tono, madaling salita, mga 300 salita. Walang headings, walang markdown."
        ),
    },

    "ceb": {
        "system": """Ikaw nagsulat og simple nga pahimangno bahin sa usa ka kusog nga bagyo para madungog sa mga tawo.

ESTILO — PURO CEBUANO, WALAY ENGLISH:
- Pagsulat sama sa imong gisulti sa imong silingan — simple, natural, dili komplikado
- BAWAL ang mga pulong nga Ingles gawas sa mga pangalan sa tawo ug lugar (hal. Pepito, Catanduanes, Luzon)
- Gamita ang inadlaw-adlaw nga Cebuano — dili pormal, dili opisyal, dili balita sa TV
- Kung adunay teknikal nga pulong, i-spell phonetically gamit ang gitling:
    tai-pun, tro-pi-kal di-pre-syon, sig-nal nam-ber wan, ki-lo-me-tros sa usa ka oras,
    storm serj, land-pol, kos-tal, pag-asa, pore-kast, wor-ning, ad-bay-so-ri
- Mubo nga mga sentence. Sayon nga mga pulong. Kinahanglan masabtan sa tanan nga Pilipino.
- Gamita ang mga digit para sa mga numero (hal. 25 kilometros sa usa ka oras, Signal 2)
- Isulti: unsa ang bagyo, asa kini, asa padulong, kinsa ang maapektuhan, unsa ang buhaton
- AYAW pagdugang og impormasyon nga wala sa orihinal nga bulletin

FORMATTING: Natural nga daloy sa prosa. Walay headings, walay bullets, walay bold, walay markdown. Blank lines tali sa mga paragraph.

GITAS-ON: Mga 300 ka pulong.""",

        "user_template": (
            "I-convert ang PAGASA weather bulletin nga kini ngadto sa simple nga pahimangno sa Cebuano.\n\n"
            "{markdown}\n\n"
            "Isulat ang pahimangno karon. Puro Cebuano — walay English gawas sa mga pangalan. "
            "Natural nga tono, sayon nga mga pulong, mga 300 ka pulong. Walay headings, walay markdown."
        ),
    },
}


def build_user_prompt(bulletin: dict, language: str) -> str:
    """Build the user prompt using the full markdown bulletin text as input."""
    template = PROMPTS[language]["user_template"]
    return template.format(markdown=bulletin["markdown"])


print("\u2713 Prompts defined for 3 languages")
for lang, prompts in PROMPTS.items():
    print(f"  {lang}: system={len(prompts['system'])} chars, template={len(prompts['user_template'])} chars")


✓ Prompts defined for 3 languages
  en: system=1205 chars, template=264 chars
  tl: system=1152 chars, template=254 chars
  ceb: system=1156 chars, template=275 chars


## 3. Generate Radio Bulletins — All Three Languages

Call Gemma 4 for each bulletin in English, Tagalog, and Cebuano (6 total scripts).

In [4]:
def generate_radio_bulletin(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a radio broadcast script for one bulletin in the specified language."""
    stem = bulletin["stem"]
    lang_names = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}
    print(f"\nGenerating: {stem} ({lang_names[language]})")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": PROMPTS[language]["system"]},
            {"role": "user", "content": build_user_prompt(bulletin, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    elapsed = time.time() - t_start

    script = response.json().get("message", {}).get("content", "").strip()

    # Word count (strip markdown syntax for accurate spoken-word estimate)
    import re
    plain = re.sub(r"[#*_`>\-]", "", script)
    word_count = len(plain.split())
    reading_minutes = word_count / 150  # ~150 wpm for radio

    # Save as markdown file
    out_path = output_dir / f"{stem}_radio_{language}.md"
    out_path.write_text(script, encoding="utf-8")

    print(f"  \u2713 Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}  |  Est. reading time: {reading_minutes:.1f} min")
    print(f"  Saved \u2192 {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "script": script,
        "word_count": word_count,
        "reading_minutes": reading_minutes,
        "elapsed": elapsed,
    }


results = []
for bulletin in bulletins:
    for lang in ["en", "tl", "ceb"]:
        result = generate_radio_bulletin(bulletin, lang)
        results.append(result)

print(f"\n\u2713 Done \u2014 {len(results)} scripts generated ({len(bulletins)} bulletins \u00d7 3 languages)")



Generating: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 14.5s
  Words: 209  |  Est. reading time: 1.4 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_en.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 26.9s
  Words: 245  |  Est. reading time: 1.6 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_tl.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 30.3s
  Words: 201  |  Est. reading time: 1.3 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_ceb.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 26.1s
  Words: 203  |  Est. reading time: 1.4 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_en.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 28.6s
  Words: 220  |  Est. reading time: 1.5 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_tl.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 30.2s
  Words: 241  |  Est. reading time: 1.6 min
  Saved → PAGASA_22-TC02_Basyan

## 6. TTS Plain Text Generation — All Three Languages

Generate TTS-optimized plain text versions of all radio scripts via a second
Gemma4 prompt. These files feed directly into notebook 08 — no markdown stripping needed.

**Output:** `data/radio_bulletins/{stem}_tts_{lang}.txt` (CEB, TL, EN)

In [ ]:
TTS_PROMPTS = {
        "en": {
            "system": (
                "You are converting a PAGASA severe weather announcement into plain text for text-to-speech synthesis.\n\n"
                "AUDIENCE: Filipinos with low literacy, limited education, and no English background. "
                "Keep the language simple. Short sentences. Common words only.\n\n"
                "RULES:\n"
                "- NO markdown: no headings (#), no bullet points (-), no asterisks (*), no bold/italic\n"
                "- NO placeholders. Never write [station name], [insert...], [your location], or anything in brackets.\n"
                "- NO radio show language. No 'Good morning listeners', no sign-offs, no station IDs.\n"
                "- Rewrite as natural flowing prose — paragraph breaks (blank lines) for pausing\n"
                "- Use simple, short words. If the original uses a complex word, use a simpler one.\n"
                "- DO NOT add any information that was not in the original script\n"
                "- Output: plain text only, no markup or formatting characters"
            ),
            "user_template": (
                "Read this markdown weather announcement and rewrite it as TTS-ready plain English text.\n\n"
                "{markdown}\n\n"
                "Write the plain English text now. Simple words. Short sentences. "
                "Paragraph breaks (blank lines) for natural pausing. No markdown. No placeholders."
            ),
        },
        "tl": {
            "system": (
                "Ikaw ay nagko-convert ng PAGASA severe weather announcement sa plain text para sa text-to-speech synthesis.\n\n"
                "AUDIENCE: Mga Pilipinong may mababang literacy, limitadong edukasyon, at walang English background. "
                "Gumamit ng simple na wika. Maikling mga pangungusap. Mga karaniwang salita lamang.\n\n"
                "PINAKAMAHALAGANG PANUNTUNAN:\n"
                "WALANG INGLES. BAWAT salitang Ingles na makikita mo sa script ay DAPAT palitan ng Tagalog o ng phonetically spelled na anyo. "
                "Ang tanging pagbubukod ay mga pangalan ng tao at lugar (hal. 'Pepito', 'Catanduanes', 'Isabela').\n\n"
                "WALANG PLACEHOLDER. Huwag isulat ang [pangalan ng istasyon], [ilagay...], [iyong lokasyon], o anumang nasa brackets.\n\n"
                "WALANG RADIO SHOW NA WIKA. Walang 'Magandang umaga mga tagapakinig', walang sign-offs, walang station IDs.\n\n"
                "MANDATORY NA PHONETIC SPELLINGS — gamitin ang mga ito palagi, hindi ang Ingles:\n"
                "  - Tropical Depression → tro-pi-kal di-pre-syon\n"
                "  - Tropical Storm → tro-pi-kal storm\n"
                "  - Severe Tropical Storm → se-beer tro-pi-kal storm\n"
                "  - Typhoon → tai-pun\n"
                "  - Super Typhoon → su-per tai-pun\n"
                "  - PAGASA / PAG-ASA → pag-asa\n"
                "  - forecast → pore-kast\n"
                "  - advisory → ad-bay-so-ri\n"
                "  - bulletin → bu-le-tin\n"
                "  - warning → wor-ning\n"
                "  - update → ap-deyt\n"
                "  - Signal Number One / Two / Three / Four / Five → sig-nal nam-ber wan / tu / tri / por / payb\n"
                "  - kilometers per hour / kph / km/h → ki-lo-me-tro ba-wat o-ras\n"
                "  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west\n"
                "  - north / south / east → nor / sow / ist\n"
                "  - northern / southern / eastern / western → nor-dern / sow-dern / is-tern / wes-tern\n"
                "  - Low Pressure Area / LPA → mababang presyon\n"
                "PARA SA MGA NUMERO — gamita ang Filipino/Spanish na mga salita:\n"
                "  - 25 km/h → beinte singko ki-lo-me-tro ba-wat o-ras\n"
                "  - 65 km/h → sisenta y singko ki-lo-me-tro ba-wat o-ras\n"
                "  - 95 km/h → nobenta y singko ki-lo-me-tro ba-wat o-ras\n"
                "  - 120 km/h → isang daan at dalawampu ki-lo-me-tro ba-wat o-ras\n"
                "  - 130 km/h → isang daan at tatlumpu ki-lo-me-tro ba-wat o-ras\n"
                "  - 150 km/h → isang daan at limampu ki-lo-me-tro ba-wat o-ras\n"
                "  - 200 km/h → dalawang daan ki-lo-me-tro ba-wat o-ras\n"
                "  - Para sa iba pang numero: singko=5, diyes=10, kinse=15, beinte=20,\n"
                "    treynta=30, kuwarenta=40, singkwenta=50, sisenta=60,\n"
                "    sitenta=70, otsenta=80, nobenta=90, isang daan=100\n\n"
                "  - hPa → ek-to-pas-kal\n"
                "  - coastal → kos-tal\n"
                "  - landfall → land-pol\n"
                "  - storm surge → storm serj\n"
                "  - flash flood → plash plud\n"
                "  - emergency → i-mer-chen-si\n"
                "  - evacuation → i-bak-yu-ey-syon\n"
                "  - center → sen-ter\n"
                "  - official → o-pi-syal\n"
                "  - Luzon → lu-son\n"
                "  - Visayas → bi-sa-yas\n"
                "  - Mindanao → min-da-naw\n\n"
                "IBA PANG PANUNTUNAN:\n"
                "- WALANG markdown: walang # headings, walang - bullets, walang * bold/italic\n"
                "- Isulat bilang natural na daloy ng prosa na angkop para basahin nang malakas\n"
                "- Panatilihin ang paragraph structure: blank lines sa pagitan ng mga paragraph\n"
                "- HUWAG magdagdag ng anumang texto na wala sa orihinal na script\n"
                "- Output: plain text lamang"
            ),
            "user_template": (
                "Basahin ang markdown weather announcement na ito at isulat muli ito bilang TTS-ready plain Tagalog text.\n\n"
                "{markdown}\n\n"
                "TANDAAN: Tagalog lamang — WALANG INGLES maliban sa mga pangalan ng tao at lugar. "
                "Gamitin ang phonetically spelled na anyo para sa lahat ng teknikal na termino. "
                "Walang placeholder. Walang radio show na wika. "
                "Paragraph breaks (blank lines) para sa natural na pausing. Walang markdown."
            ),
        },
        "ceb": {
            "system": (
                "Ikaw kay mag-convert sa PAGASA severe weather announcement ngadto sa plain text para sa text-to-speech synthesis.\n\n"
                "AUDIENCE: Mga Pilipino nga di kaayo literate, limitado ang edukasyon, ug gamay ra ang English background. "
                "Gamita ang simple nga pinulongan. Mubo nga mga sentence. Komon nga mga pulong lamang.\n\n"
                "PINAKA-IMPORTANTE NGA LAGDA:\n"
                "WALAY ENGLISH. ANG MATAG English word nga imong makita sa script KINAHANGLAN ilisdan sa Cebuano o sa phonetically spelled nga porma. "
                "Ang bugtong eksepsyon mao ang mga pangalan sa tawo ug lugar (sampol. 'Pepito', 'Catanduanes', 'Isabela').\n\n"
                "ug ang mga Proper Nouns nga dili ma-translate (sampol. 'PAGASA', 'Signal Number One').\n\n"
                "WALAY PLACEHOLDER. Ayaw isulat ang [ngalan sa istasyon], [ibutang...], [imong lokasyon], o bisan unsa nga anaa sa brackets.\n\n"
                "WALAY RADIO SHOW NGA PULONG. Walay 'Maayong buntag mga tigpaminaw', walay sign-offs, walay station IDs.\n\n"
                "MANDATORY NGA PHONETIC SPELLINGS — gamita kini kanunay, dili ang English:\n"
                "  - Tropical Depression → tro-pi-kal di-pre-syon\n"
                "  - Tropical Storm → tro-pi-kal storm\n"
                "  - Severe Tropical Storm → se-beer tro-pi-kal storm\n"
                "  - Typhoon → tai-pun\n"
                "  - Super Typhoon → su-per tai-pun\n"
                "  - PAGASA / PAG-ASA → pag-asa\n"
                "  - forecast → pore-kast\n"
                "  - advisory → ad-bay-so-ri\n"
                "  - bulletin → bu-le-tin\n"
                "  - warning → wor-ning\n"
                "  - update → ap-deyt\n"
                "  - Signal Number One / Two / Three / Four / Five → sig-nal nam-ber wan / tu / tri / por / payb\n"
                "  - kilometers per hour / kph / km/h → ki-lo-me-tros sa usa ka oras\n"
                "  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west\n"
                "  - north / south / east → nor / sow / ist\n"
                "  - northern / southern / eastern / western → nor-dern / sow-dern / is-tern / wes-tern\n"
                "  - Low Pressure Area / LPA → ubos nga presyon\n"
                "PARA SA MGA NUMERO — gamita ang Cebuano/Spanish nga mga pulong:\n"
                "  - 25 km/h → baynte singko ki-lo-me-tros sa usa ka oras\n"
                "  - 65 km/h → sisenta y singko ki-lo-me-tros sa usa ka oras\n"
                "  - 95 km/h → nobenta y singko ki-lo-me-tros sa usa ka oras\n"
                "  - 120 km/h → isyento baynte ki-lo-me-tros sa usa ka oras\n"
                "  - 130 km/h → isyento treynta ki-lo-me-tros sa usa ka oras\n"
                "  - 150 km/h → isyento singkwenta ki-lo-me-tros sa usa ka oras\n"
                "  - 200 km/h → dos siyentos ki-lo-me-tros sa usa ka oras\n"
                "  - Para sa ubang numero: singko=5, diyes=10, kinse=15, baynte=20,\n"
                "    treynta=30, kuwarenta=40, singkwenta=50, sisenta=60,\n"
                "    sitenta=70, otsenta=80, nobenta=90, isyento=100\n\n"
                "  - hPa → ek-to-pas-kal\n"
                "  - coastal → kos-tal\n"
                "  - landfall → land-pol\n"
                "  - storm surge → storm serj\n"
                "  - flash flood → plash plud\n"
                "  - emergency → i-mer-chen-si\n"
                "  - evacuation → i-bak-yu-ey-syon\n"
                "  - center → sen-ter\n"
                "  - official → o-pi-syal\n"
                "  - Luzon → lu-son\n"
                "  - Visayas → bi-sa-yas\n"
                "  - Mindanao → min-da-naw\n\n"
                "UBAN PA NGA MGA LAGDA:\n"
                "- WALAY markdown: walay # headings, walay - bullets, walay * bold/italic\n"
                "- Isulat isip natural nga daloy sa prosa nga angay basahon sa makusog\n"
                "- Pahimusa ang paragraph structure: blank lines tali sa mga paragraph\n"
                "- AYAW pagdugang og bisan unsa nga texto nga wala sa orihinal nga script\n"
                "- Output: plain text lamang"
            ),
            "user_template": (
                "Basaha kining markdown weather announcement ug isulat kini pag-usab isip TTS-ready plain Cebuano text.\n\n"
                "{markdown}\n\n"
                "HINUMDUMI: Cebuano lamang — WALAY ENGLISH gawas sa mga pangalan sa tawo ug lugar. "
                "Gamita ang phonetically spelled nga porma para sa tanan nga teknikal nga termino. "
                "Walay placeholder. Walay radio show nga pulong. "
                "Paragraph breaks (blank lines) para sa natural nga pausing. Walay markdown."
            ),
        },
    }


def build_tts_prompt(markdown_script: str, language: str) -> str:
    """Build the user prompt for TTS text generation."""
    return TTS_PROMPTS[language]["user_template"].format(markdown=markdown_script)


print("\u2713 TTS_PROMPTS defined for CEB, TL, EN")
print(f"  ceb: system={len(TTS_PROMPTS['ceb']['system'])} chars")
print(f"  tl:  system={len(TTS_PROMPTS['tl']['system'])} chars")
print(f"  en:  system={len(TTS_PROMPTS['en']['system'])} chars")

In [6]:
def generate_tts_text(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a TTS-ready plain text script for one bulletin."""
    stem = bulletin["stem"]
    lang_names = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}
    print(f"\nGenerating TTS text: {stem} ({lang_names[language]})")

    markdown_script = (output_dir / f"{stem}_radio_{language}.md").read_text(encoding="utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": TTS_PROMPTS[language]["system"]},
            {"role": "user", "content": build_tts_prompt(markdown_script, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    elapsed = time.time() - t_start

    tts_text = response.json().get("message", {}).get("content", "").strip()

    out_path = output_dir / f"{stem}_tts_{language}.txt"
    out_path.write_text(tts_text, encoding="utf-8")

    word_count = len(tts_text.split())
    print(f"  ✓ Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}")
    print(f"  Saved → {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "tts_text": tts_text,
        "word_count": word_count,
        "elapsed": elapsed,
    }


# Generate TTS text for both bulletins — all three languages
tts_results = []
for bulletin in bulletins:
    for lang in ["ceb", "tl", "en"]:
        result = generate_tts_text(bulletin, lang)
        tts_results.append(result)

print(f"\n✓ Done — {len(tts_results)} TTS text files generated")

# Preview first bulletin (CEB)
print("\nPREVIEW — CEB TTS text (first 500 chars)")
print("=" * 60)
ceb_result = [r for r in tts_results if r["language"] == "ceb"][0]
print(ceb_result["tts_text"][:500])
print("...")


Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 37.8s
  Words: 201
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_ceb.txt

Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 27.5s
  Words: 245
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_tl.txt

Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 24.9s
  Words: 209
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_en.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 25.2s
  Words: 241
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_ceb.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 23.8s
  Words: 220
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_tl.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 18.1s
  Words: 203
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_en.txt

✓ Done — 6 TTS text files generated

PREVIEW — CEB TTS text (first 500 chars)
Mga kaigsoonan, pagtagad og pagpamati

## 4. Review Output

Display each generated script for manual inspection (grouped by bulletin, showing all languages).

In [9]:
from IPython.display import display, Markdown
from collections import defaultdict

grouped = defaultdict(list)
for r in results:
    grouped[r["stem"]].append(r)

lang_labels = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}

for stem, versions in grouped.items():
    display(Markdown(f"---\n# {stem}\n---"))
    for version in versions:
        lang = version["language"]
        meta = (
            f"**Language:** {lang_labels[lang]}  \u2502  "
            f"**Words:** {version['word_count']}  \u2502  "
            f"**Est. reading time:** {version['reading_minutes']:.1f} min"
        )
        display(Markdown(f"## {lang_labels[lang]} Version\n\n{meta}\n\n---\n\n{version['script']}"))


---
# PAGASA_20-19W_Pepito_SWB#01
---

## English Version

**Language:** English  │  **Words:** 209  │  **Est. reading time:** 1.4 min

---

A tropical depression named Pepito has formed east of ka-tan-du-a-nes. Right now, it is still developing, and meteorologists think it could get stronger into a tropical storm or even a typhoon.

Pepito is expected to move generally west-northwestward or north-northwestward today. Later, it will turn more toward the west, heading for northern Luzon. The forecast shows Pepito might make landfall on the eastern coast of northern Luzon on the afternoon of October 21.

The areas expected to be hit the hardest are ka-tan-du-a-nes, northern sa-mar, and sa-mar itself. Many areas in Luzon will expect gusty winds and heavy rain. Be ready for potential heavy rains in the bi-sa-yas region, especially tonight.

Several areas need to be careful. These include the bi-kol region and ka-tan-du-a-nes. Other areas are northern sa-mar, sa-mar, southern ley-te, mas-ba-te, and various parts of que-son. Along the coast, people in i-sa-be-la, ka-ga-yan, and pangasinan also need to prepare.

If you are on the water, expect moderate to strong seas. Strong winds and heavy rain are expected in many parts of the Visayas and the eastern coasts.

Officials are telling everyone to take proper safety actions now. We are advised to stay alert and keep listening for the next severe weather update, as the situation is changing.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 245  │  **Est. reading time:** 1.6 min

---

May bagong update po tayo tungkol sa bagyo na si Pepito.

Ang pag-iingat po muna sa ating lahat. Ang low pressure area na nasa silangan ng Catanduanes ay lumaganap na at naging Tropical Depression na ang pangalan ay Pepito. Ayon sa mga pag-aaral, aakyat pa nga raw ito ng lakas—baka maging tropical storm o mas malala pa.

Ang direksyon po ni Pepito ay lalakas pa at magtutuloy papunta sa kanluran at hilagang-kanluran. Inaasahan pong tumama ang bagyo sa baybayin ng Northern Luzon sa hapon ng 21 Oktubre.

Sino po ang unang maaapektuhan? Ang mga lugar na dapat maging handa ay Catanduanes, Northern Samar, Samar, at ang mga coastal area ng Northern Luzon. Pero hindi lang po sila. Inaasahan po ring bumagsak ang ulan at magkaroon ng malakas na hangin sa Bicol Region, sa mga bahagi ng Quezon, at sa buong kabisera ng Northern Luzon.

Dahil po sa lakas ng ulan at hangin, malalakas at mataas din po ang mga alon sa karagatan, hanggang 4.5 metro raw. Kaya naman po, huwag pong maglayag o magbiyahe sa tubig-tabang malapit sa daluyan ng tubig.

Ang payo po namin ay maging handa sa anumang pagbabago. Patuloy pong mag-ingat sa matitindig na ulan, malakas na hangin, at pagtaas ng tubig sa mga mababang lugar. Manatiling nakatutok sa mga bagong *wor-ning* o anunsyo mula sa mga awtoridad. Paghandaan po ang lahat ng pamilya at huwag pong magpabaya sa mga gamit na pang-emergency. Kailangan po nating maging handa sa pagdating ni Pepito.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 201  │  **Est. reading time:** 1.3 min

---

Mga kaigsoonan, pagtagad og pagpamati sa maayo. Aduna kitay bag-ong pahibalo bahin sa kusog nga bagyo nga gitawag og "Pepito." Karon na kini usa ka *Tropical Depression* nga nahitabo sa pagkasad-anan sa Catanduanes.

Pag-ingon pa niini ang mga opisyal, ang "Pepito" moagi pa sa kasad-anan paingon kini paingon sa amihanan ug sa pagkasad-anan sa Luzon. Ang gi-ingon nga dalan mao nga mahabot kini sa baybayon sa Northern Luzon sa pagkahapon sa adlaw nga Oktubre 21.

Ang labing mag-ayo apil niini mao ang mga lugar sa Catanduanes, Northern Samar, Samar, Southern Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, ug mga bahin sa Quezon. Sa atong mga baybayon sa Northern Luzon, Isabela, Cagayan, ug Bataan, aduna usab moyaog.

Sa tanan nga maapektuhan, mag-andam og kusog nga pag-ulan ug mo-i-hiway nga mga hangin. Ang dagat mahimo nga grabe, nga moabot og 2.5 hangtud 4.5 ka metros. Busa, kinahanglan nga mag-amping kon mo-sakay mo sa bisan unsang sakyanan nga sa dagat, ilabi na sa mga bantay-baybayon.

Ang labing importante mao ang pagka-andam. Pangitaa ang labing luwas nga dapit sa inyong panimalay. Pagplano og pag-ampo ug pag-andam og emergency kit. Kanunay kining pag-monitor og mga sunod nga *ad-bay-so-ri* gikan sa PAGASA. Pagpabilin nga mag-amping, mga kaigsoonan.

---
# PAGASA_22-TC02_Basyang_TCA#01
---

## English Version

**Language:** English  │  **Words:** 203  │  **Est. reading time:** 1.4 min

---

We are tracking Tropical Storm "MALAKAS." Right now, the center of the storm is located about 2 kilometers east of Mindanao. It is moving west northwest at about 15 kilometers per hour.

The storm has maximum sustained winds of 75 kilometers per hour near its center, with gusts possibly reaching 90 kilometers per hour.

Looking at the forecast, MALAKAS is expected to continue moving steadily. Importantly, its entire predicted path shows it staying outside the Philippine Area of Responsibility. The storm is tracking generally northeastward, passing far to the east of the main islands, including the areas near Central Luzon and Northern Luzon.

Because the storm is currently positioned and forecast to remain outside the populated areas, the immediate threat to communities in ka-ga-yan, i-sa-be-la, or anywhere in the bi-kol region is low.

However, we are still watching this storm closely. As it continues to move and potentially increase in strength, people should stay aware of the situation. We are advising everyone to keep monitoring local official announcements for the latest updates. Please remember to prepare your families and your homes in case any changes in the storm's track occur. We will continue to provide updates as soon as new information is available.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 220  │  **Est. reading time:** 1.5 min

---

Mga kapitbahay, may ulat tayo mula sa PAGASA tungkol sa bagyo. Ang tawag dito ay Tropical Storm na "MALAKAS."

Sa ngayon, ang sentro ng bagyo ay nasa silangan ng Mindanao, kaya wala pa po itong epekto sa ating mga isla dahil nasa labas pa ito ng ating lugar ng responsibilidad.

Ayon sa pagtataya, ang lakas ng "MALAKAS" ay may hangin na umaabot sa 75 kilometro bawat oras, at maaaring umabot pa ito ng hangin na 90 kilometro bawat oras. Naglalakbay ito papunta sa kanluran-taang-kanluran sa bilis na mga 15 kilometro bawat oras.

Pero tandaan po natin, habang malakas ang bagyong ito, itinuturo ng pagtataya na patuloy itong liliko at aalis pa sa atin. Sa mga darating na araw, dadaan po ito sa kanluran, at patuloy itong lilipat patungong silangan at paakyat pa sa hilaga.

Kaya kahit na sa mga forecast, hindi ito ipinapalagay na tatama sa Pilipinas. Magpapatuloy lang po itong paglihip sa dagat, malayo sa Visayas, at malapit na sa Luzon.

Ang mahalaga po ay, manatiling handa tayo. Kahit na malayo pa ito, laging nakakaapekto ang mga bagyo sa klima at karagatan. Manatiling nakatutok sa mga abiso ng lokal na pamahalaan. Maghanda tayo ng mga pangangailangan at sundin ang mga magiging utos, dahil baka may maging pagbabago sa mga bagyo sa mga susunod na araw. Manatiling ligtas tayong lahat.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 241  │  **Est. reading time:** 1.6 min

---

Mga kaigsoonan, paminaw mo og maayo kay aduna tay bag-ong ad-bay-so-ri gikan sa mga awtoridad bahin sa usa ka tro-pi-kal di-pre-syon.

Ang nagngalan niini kay Tropikal nga Bagyo nga “MALAKAS”. Sa pagkahapon, pagkakaron, anaa ni kini sa gawas sa atong Philippine Area of Responsibility.

Base sa pagtan-aw sa ilang mga mapa, ang “MALAKAS” nagkupot og kusog pa gihapon, nga ang hangin mahimo nga moabot og 75 kilometros sa usa ka oras, ug mohatag og mga pag-agaw nga moabot og 90 kilometros sa usa ka oras. Karon pa, mo-uswag ni siya pag-west-northwest sa kalit nga pace nga 15 kilometros sa usa ka oras.

Importante nga mahibal-an ninyo nga pag-analisar sa ilang forecast track, ang bagyo moagi ra sa gawas sa atong mga isla. Paglabay sa Mindanao, dayon sa Visayas, paglabay sa habagatan nga bahin sa Luzon, ug hangtud moabot siya sa gamay nga pag-atras sa amoy sa amoa sa Northern Luzon.

Busa, bisan pa nga duol kaayo ni siya, ang among mga pag-forecast nagpakita nga dili kini direkta makakaigo o makapasagol sa atong yuta.

Apan, dili pa sab kini pasidaan. Ang pag-monitor kanunay mao pa gihapon ang importante. Palihug pag-andam ang inyong mga pamilya, ug siguruha nga naa moy emergency kit. Pag-andam sa mga mahinaw nga pag-ampo, ug pag-amping sa mga dagat.

Pagpabilin sa pagpamatuod sa inyong mga maayo nga kasayuran gikan sa mga opisyal, dili sa mga silinganan lang. Pagbantay gihapon sa inyong mga palibot. Daghang salamat ug pag-amping kita tanan.

## 5. Length Check

Verify word counts across all three languages are close to the 300-word target. If any are significantly short or long, we can adjust the prompts.

In [10]:
TARGET_WORDS = 300
WORDS_PER_MIN = 150

print("\nLENGTH SUMMARY")
print("=" * 60)
print(f"{'Bulletin':<35} {'Lang':>4} {'Words':>6}  {'Minutes':>7}  {'vs Target':>10}")
print("-" * 60)

for result in results:
    diff = result['word_count'] - TARGET_WORDS
    diff_str = f"+{diff}" if diff >= 0 else str(diff)
    label = result['stem'].replace('PAGASA_', '')
    lang = result['language'].upper()
    print(f"{label:<35} {lang:>4} {result['word_count']:>6}  {result['reading_minutes']:>6.1f}m  {diff_str:>10}")

print("-" * 60)
print(f"Target: {TARGET_WORDS} words ≈ {TARGET_WORDS/WORDS_PER_MIN:.0f} minutes at {WORDS_PER_MIN} wpm")
print(f"\n✓ {len(results)} scripts saved to: {output_dir.absolute()}")


LENGTH SUMMARY
Bulletin                            Lang  Words  Minutes   vs Target
------------------------------------------------------------
20-19W_Pepito_SWB#01                  EN    209     1.4m         -91
20-19W_Pepito_SWB#01                  TL    245     1.6m         -55
20-19W_Pepito_SWB#01                 CEB    201     1.3m         -99
22-TC02_Basyang_TCA#01                EN    203     1.4m         -97
22-TC02_Basyang_TCA#01                TL    220     1.5m         -80
22-TC02_Basyang_TCA#01               CEB    241     1.6m         -59
------------------------------------------------------------
Target: 300 words ≈ 2 minutes at 150 wpm

✓ 6 scripts saved to: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins
